# Gerador de Cartas de Magic: The Gathering

Projeto da disciplina de IA e Bibliotecas de Dados para Python.

**Como usar:** no menu acima, clique em **Ambiente de execução > Executar tudo** (ou `Ctrl+F9`). As duas primeiras células preparam o ambiente (clonam o repositório e instalam as dependências); a terceira mostra o formulário para gerar a carta.

O código completo do pipeline de geração está em [`src/pipeline/generator.py`](../src/pipeline/generator.py) no repositório — este notebook só importa e chama a função `gerar_carta`, para que a mesma lógica possa também ser usada por outras interfaces no futuro (ex: um app web).

## 1. Preparar o ambiente

Clona o repositório (se ainda não estiver presente) e instala as dependências.

In [ ]:
import os

REPO_URL = "https://github.com/funkymonks21/MTGCardCreator-IFRS.git"
REPO_DIR = "MTGCardCreator-IFRS"

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL}
else:
    # a pasta ja existe de uma execucao anterior nesta sessao do Colab --
    # puxa os commits novos em vez de ignorar
    !cd {REPO_DIR} && git pull -q

%cd {REPO_DIR}
!pip install -q -r requirements.txt

import sys
sys.path.insert(0, "src")

import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("punkt_tab", quiet=True)

print("Ambiente pronto.")

## 2. Carregar o pipeline

Importa a função `gerar_carta`. Nas próximas etapas do projeto, esta célula também vai carregar os modelos pré-treinados (Word2Vec, regressor de custo de mana, classificador PyTorch) a partir da pasta `models/`, para que ninguém precise treinar nada do zero.

In [ ]:
from pipeline import gerar_carta, COLOR_PIE

# TODO (próximas etapas): carregar aqui os artefatos pré-treinados, ex:
# modelo_mtg = Word2Vec.load("models/word2vec_mtg.model")
# modelo_custo = joblib.load("models/regressor_custo.pkl")
# classificador_cor = torch.load("models/classificador_cor.pt")

print("Pipeline carregado. Cores disponíveis:", list(COLOR_PIE))

### Teste rápido (sem usar os campos acima)

Alguns exemplos que mostram a inferência de mecânica + o ajuste pelo color pie em ação:

In [ ]:
exemplos = [
    ("Murder", "Azul"),      # "murder" infere "destroy" -> Azul nao tem destroy -> vira bounce
    ("Murder", "Preto"),     # Preto tem destroy -> mantem destroy
    ("Lightning Bolt", "Vermelho"),
    ("jasdhahs12123", "Verde"),  # nome sem sentido -> cai no fallback padrao
]

for nome_exemplo, cor_exemplo in exemplos:
    c = gerar_carta(nome_exemplo, cor_exemplo)
    print(f"{nome_exemplo!r} + {cor_exemplo}: "
          f"mecanica inferida='{c.mecanica_inferida}' (via {c.origem_inferencia}), "
          f"mecanica final='{c.mecanica_final}'")
    print(f"  -> {c.texto_regras}\n")

## 3. Gerar sua carta

Preencha o nome e escolha a cor, depois clique em **Gerar carta**.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

campo_nome = widgets.Text(
    description="Nome:",
    placeholder="ex: Murder, Lightning Bolt, jasdhahs12123...",
    style={"description_width": "initial"},
)
campo_cor = widgets.Dropdown(
    options=list(COLOR_PIE.keys()),
    description="Cor:",
    style={"description_width": "initial"},
)
botao_gerar = widgets.Button(description="Gerar carta", button_style="success")
saida = widgets.Output()


def ao_clicar(_botao):
    with saida:
        clear_output(wait=True)
        carta = gerar_carta(campo_nome.value, campo_cor.value)
        display(carta.imagem_final)
        print(f"Nome: {carta.nome}")
        print(f"Custo de mana: {carta.custo_mana}")
        print(f"Tipo: {carta.tipo}")
        print(f"Texto: {carta.texto_regras}")
        print()
        print(f"[debug] mecânica inferida do nome: {carta.mecanica_inferida} (via {carta.origem_inferencia})")
        if carta.mecanica_final != carta.mecanica_inferida:
            print(f"[debug] ajustada para '{carta.mecanica_final}' por causa do color pie de {carta.cor}")


botao_gerar.on_click(ao_clicar)
display(campo_nome, campo_cor, botao_gerar, saida)